# Group 3 — TobesureMarket: a Polymarket-style prediction market on Sepolia

**Course:** Blockchain & Decentralized Systems · IE MDBI
**Goal:** implement, deploy and test a binary (YES/NO) prediction-market smart contract on the Ethereum **Sepolia** testnet, following the flow of `6_Deploying_to_SepoliaTestnet_CreatingContract_v2.ipynb`.

| Step | What happens | Proof |
|---|---|---|
| 1 | Connect to Sepolia through **Infura** | chain id 11155111 |
| 2 | Compile `TobesureMarket.sol` with `solc 0.8.20` | ABI + bytecode |
| 3 | **Deploy** the contract | tx hash + Etherscan link |
| 4 | **Create a market** | `MarketCreated` event |
| 5 | **Every group member bets** YES or NO | `BetPlaced` events |
| 6 | **Resolve** the market | `MarketResolved`, `Payout`, `FeePaid` events |
| 7 | Verify payouts & export the transaction log for the slides | `tx_log.csv` |

> ⚠️ **Security:** private keys are read from a local `keys.env` file and are **never** printed or hard-coded. Use **testnet-only** wallets. Never put a key that holds real ETH in this notebook, and never commit `keys.env`.

## 0 · Setup

Install once (uncomment). `eth-tester[py-evm]` is only needed for the optional **local rehearsal mode**.

In [ ]:
# !pip install web3 py-solc-x python-dotenv pandas
# !pip install "eth-tester[py-evm]"   # optional: offline rehearsal mode

Create a file called **`keys.env`** in the same folder as this notebook:

```
INFURA_API_KEY=your_infura_project_id
DEPLOYER_PRIVATE_KEY=0x...          # wallet that deploys + creates + resolves (the "platform")

# Optional — one line per group member who bets from THIS laptop (testnet-only wallets!)
MEMBER_1_NAME=Max
MEMBER_1_PRIVATE_KEY=0x...
MEMBER_2_NAME=...
MEMBER_2_PRIVATE_KEY=0x...

# After the first deployment, paste the address here so re-runs reuse the same contract
CONTRACT_ADDRESS=
```

Fund each wallet with Sepolia ETH from a faucet (e.g. Google Cloud Web3 faucet, Alchemy or Infura Sepolia faucet). ~0.05 SepoliaETH is plenty for the deployer; ~0.01 per member.

In [2]:
import os, json, time
from datetime import datetime, timezone

import pandas as pd
from dotenv import load_dotenv
from web3 import Web3
from web3.logs import DISCARD   # ignore logs from other event types when decoding
from eth_account import Account
import solcx

# Keys live in "keys.env" (visible name, so Jupyter can open it); ".env" also works as a fallback.
load_dotenv("keys.env") or load_dotenv(".env")

def load_key(name):
    # Read a private key from keys.env and normalise it (spaces, quotes, missing 0x)
    k = (os.getenv(name) or "").strip().strip('"').strip("'")
    if not k:
        return None
    return k if k.startswith("0x") else "0x" + k

# "sepolia" for the real assignment, "local" for an offline rehearsal on an in-memory chain
NETWORK_MODE = os.getenv("NETWORK_MODE", "sepolia").lower()
SOLC_VERSION = "0.8.20"
EXPLORER = "https://sepolia.etherscan.io"
print("Mode:", NETWORK_MODE)

Mode: sepolia


## 1 · Connect to Sepolia via Infura

In [3]:
if NETWORK_MODE == "sepolia":
    INFURA_API_KEY = (os.getenv("INFURA_API_KEY") or "").strip()
    assert INFURA_API_KEY, "INFURA_API_KEY fehlt in keys.env (gleicher Ordner wie dieses Notebook)"
    w3 = Web3(Web3.HTTPProvider(f"https://sepolia.infura.io/v3/{INFURA_API_KEY}"))
    assert load_key("DEPLOYER_PRIVATE_KEY"), "DEPLOYER_PRIVATE_KEY fehlt in keys.env"
    deployer = Account.from_key(load_key("DEPLOYER_PRIVATE_KEY"))
else:
    from web3 import EthereumTesterProvider
    w3 = Web3(EthereumTesterProvider())
    deployer = Account.create()
    # fund the throw-away deployer from the tester's pre-funded account
    w3.eth.send_transaction({"from": w3.eth.accounts[0], "to": deployer.address, "value": w3.to_wei(10, "ether")})

assert w3.is_connected(), "Not connected — check the Infura key / internet"
CHAIN_ID = w3.eth.chain_id
print("Connected · chain id:", CHAIN_ID, "(Sepolia = 11155111)")
print("Latest block:", w3.eth.block_number)
print("Deployer:", deployer.address)
print("Deployer balance:", w3.from_wei(w3.eth.get_balance(deployer.address), "ether"), "ETH")

Connected · chain id: 11155111 (Sepolia = 11155111)
Latest block: 12050916
Deployer: 0xd4556175F88cc8B8612177a771E75AacAB27eff9
Deployer balance: 0.05 ETH


## 2 · The smart contract — `TobesureMarket.sol`

**Business logic (pari-mutuel pool, inspired by Polymarket):**
- `createMarket(question, duration)` – anyone opens a YES/NO market; the caller becomes its **creator**.
- `placeBet(marketId, isYes)` – payable; stake test-ETH on YES or NO until the deadline.
- `yesProbabilityBps(marketId)` – implied probability = YES pool ÷ total pool (Polymarket's "price").
- `resolveMarket(marketId, outcome)` – only the market creator (or platform owner as fallback arbiter). Then **automatically**:
  1. platform fee (2%) → platform owner, 2. creator fee (1%) → market creator, 3. remaining 97% → winners pro rata.
- Outcome `Invalid` (or no one on the winning side) → **full refunds, no fees**.
- Failed transfers are credited to `pendingWithdrawals` → `withdraw()` (one bad recipient can't block payouts).
- Safety: reentrancy guard, checks-effects-interactions, fee cap (10%), bettor cap (200) to bound gas.

In [4]:
CONTRACT_SOURCE = r"""
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.20;

/**
 * @title  TobesureMarket
 * @notice Binary (YES / NO) prediction markets paid in ETH — Group 3, Blockchain & Decentralized Systems (IE MDBI).
 *
 * Model: pari-mutuel pool (inspired by Polymarket, simplified).
 *  - Anyone can create a market with a question and a betting deadline.
 *  - Anyone can bet test ETH on YES or NO until the deadline.
 *  - The market creator (or the platform owner as fallback arbiter) resolves the outcome.
 *  - On resolution the contract automatically:
 *        1. pays the platform fee to the platform owner,
 *        2. pays the creator fee to the market creator,
 *        3. splits the remaining pool among winning bettors, pro rata to their stake.
 *  - If the outcome is INVALID, or nobody bet on the winning side, everyone is refunded and no fees are charged.
 *  - Any transfer that fails (e.g. a contract wallet that rejects ETH) is credited to a pull-withdrawal
 *    balance, so one bad recipient can never block the whole market.
 */
contract TobesureMarket {
    // ─────────────────────────────── Types ───────────────────────────────
    enum Outcome { Unresolved, Yes, No, Invalid }

    struct Market {
        string  question;
        address creator;
        uint256 deadline;      // betting closes at this timestamp
        uint256 yesPool;       // total wei staked on YES
        uint256 noPool;        // total wei staked on NO
        Outcome outcome;
        bool    resolved;
    }

    // ───────────────────────────── Constants ─────────────────────────────
    uint256 public constant BPS = 10_000;
    uint256 public constant MAX_FEE_BPS = 1_000;      // fees can never exceed 10% combined
    uint256 public constant MAX_BETTORS = 200;        // bounds the payout loop's gas cost
    uint256 public constant MIN_BET = 0.0001 ether;

    // ───────────────────────────── Storage ───────────────────────────────
    address public owner;                 // the platform
    uint256 public platformFeeBps;        // e.g. 200 = 2%
    uint256 public creatorFeeBps;         // e.g. 100 = 1%
    uint256 public marketCount;

    mapping(uint256 => Market) public markets;
    mapping(uint256 => mapping(address => uint256)) public yesStake;
    mapping(uint256 => mapping(address => uint256)) public noStake;
    mapping(uint256 => address[]) private bettors;
    mapping(uint256 => mapping(address => bool)) private isBettor;
    mapping(uint256 => mapping(address => uint256)) public paidOut;   // what each address received per market

    mapping(address => uint256) public pendingWithdrawals;            // fallback for failed pushes

    bool private locked;

    // ───────────────────────────── Events ────────────────────────────────
    event MarketCreated(uint256 indexed marketId, address indexed creator, string question, uint256 deadline);
    event BetPlaced(uint256 indexed marketId, address indexed bettor, bool isYes, uint256 amount);
    event MarketResolved(uint256 indexed marketId, Outcome outcome, uint256 totalPool, uint256 platformFee, uint256 creatorFee);
    event FeePaid(uint256 indexed marketId, address indexed to, uint256 amount, bool isPlatformFee);
    event Payout(uint256 indexed marketId, address indexed to, uint256 amount);
    event PayoutDeferred(address indexed to, uint256 amount);
    event Withdrawn(address indexed to, uint256 amount);
    event FeesUpdated(uint256 platformFeeBps, uint256 creatorFeeBps);

    // ──────────────────────────── Modifiers ──────────────────────────────
    modifier onlyOwner() {
        require(msg.sender == owner, "Not platform owner");
        _;
    }

    modifier nonReentrant() {
        require(!locked, "Reentrancy");
        locked = true;
        _;
        locked = false;
    }

    modifier marketExists(uint256 marketId) {
        require(marketId < marketCount, "Market does not exist");
        _;
    }

    // ─────────────────────────── Constructor ─────────────────────────────
    constructor(uint256 _platformFeeBps, uint256 _creatorFeeBps) {
        require(_platformFeeBps + _creatorFeeBps <= MAX_FEE_BPS, "Fees too high");
        owner = msg.sender;
        platformFeeBps = _platformFeeBps;
        creatorFeeBps = _creatorFeeBps;
        emit FeesUpdated(_platformFeeBps, _creatorFeeBps);
    }

    // ──────────────────────────── Core logic ─────────────────────────────

    /// @notice Open a new YES/NO market. Betting stays open for `durationSeconds`.
    function createMarket(string calldata question, uint256 durationSeconds) external returns (uint256 marketId) {
        require(bytes(question).length > 0, "Empty question");
        require(durationSeconds > 0, "Duration must be > 0");

        marketId = marketCount++;
        Market storage m = markets[marketId];
        m.question = question;
        m.creator = msg.sender;
        m.deadline = block.timestamp + durationSeconds;

        emit MarketCreated(marketId, msg.sender, question, m.deadline);
    }

    /// @notice Stake ETH on YES (`isYes = true`) or NO (`isYes = false`).
    function placeBet(uint256 marketId, bool isYes) external payable marketExists(marketId) {
        Market storage m = markets[marketId];
        require(!m.resolved, "Market resolved");
        require(block.timestamp < m.deadline, "Betting closed");
        require(msg.value >= MIN_BET, "Bet below minimum");

        if (!isBettor[marketId][msg.sender]) {
            require(bettors[marketId].length < MAX_BETTORS, "Bettor limit reached");
            isBettor[marketId][msg.sender] = true;
            bettors[marketId].push(msg.sender);
        }

        if (isYes) {
            m.yesPool += msg.value;
            yesStake[marketId][msg.sender] += msg.value;
        } else {
            m.noPool += msg.value;
            noStake[marketId][msg.sender] += msg.value;
        }

        emit BetPlaced(marketId, msg.sender, isYes, msg.value);
    }

    /// @notice Settle the market and distribute all funds. Callable by the market creator or the platform owner.
    function resolveMarket(uint256 marketId, Outcome outcome) external nonReentrant marketExists(marketId) {
        Market storage m = markets[marketId];
        require(msg.sender == m.creator || msg.sender == owner, "Not creator or owner");
        require(!m.resolved, "Already resolved");
        require(outcome != Outcome.Unresolved, "Invalid outcome value");

        // Effects first (checks-effects-interactions)
        m.resolved = true;
        m.outcome = outcome;

        uint256 total = m.yesPool + m.noPool;
        uint256 winningPool = outcome == Outcome.Yes ? m.yesPool : (outcome == Outcome.No ? m.noPool : 0);
        address[] storage list = bettors[marketId];

        // Refund path: INVALID outcome, or nobody backed the winning side → return stakes, no fees.
        if (outcome == Outcome.Invalid || winningPool == 0) {
            emit MarketResolved(marketId, outcome, total, 0, 0);
            for (uint256 i = 0; i < list.length; i++) {
                address b = list[i];
                uint256 stake = yesStake[marketId][b] + noStake[marketId][b];
                if (stake > 0) _pay(marketId, b, stake);
            }
            return;
        }

        uint256 platformFee = (total * platformFeeBps) / BPS;
        uint256 creatorFee = (total * creatorFeeBps) / BPS;
        uint256 distributable = total - platformFee - creatorFee;

        emit MarketResolved(marketId, outcome, total, platformFee, creatorFee);

        // Pay winners pro rata; track what was actually sent so rounding dust is accounted for.
        uint256 distributed;
        for (uint256 i = 0; i < list.length; i++) {
            address b = list[i];
            uint256 stake = outcome == Outcome.Yes ? yesStake[marketId][b] : noStake[marketId][b];
            if (stake == 0) continue;
            uint256 share = (stake * distributable) / winningPool;
            distributed += share;
            _pay(marketId, b, share);
        }

        // Rounding dust goes to the platform so the contract never keeps stray wei.
        platformFee += distributable - distributed;

        if (creatorFee > 0) {
            _sendFee(marketId, m.creator, creatorFee, false);
        }
        if (platformFee > 0) {
            _sendFee(marketId, owner, platformFee, true);
        }
    }

    /// @notice Pull any payout that could not be pushed automatically.
    function withdraw() external nonReentrant {
        uint256 amount = pendingWithdrawals[msg.sender];
        require(amount > 0, "Nothing to withdraw");
        pendingWithdrawals[msg.sender] = 0;
        (bool ok, ) = payable(msg.sender).call{value: amount}("");
        require(ok, "Withdraw failed");
        emit Withdrawn(msg.sender, amount);
    }

    // ───────────────────────────── Admin ─────────────────────────────────
    function setFees(uint256 _platformFeeBps, uint256 _creatorFeeBps) external onlyOwner {
        require(_platformFeeBps + _creatorFeeBps <= MAX_FEE_BPS, "Fees too high");
        platformFeeBps = _platformFeeBps;
        creatorFeeBps = _creatorFeeBps;
        emit FeesUpdated(_platformFeeBps, _creatorFeeBps);
    }

    function transferOwnership(address newOwner) external onlyOwner {
        require(newOwner != address(0), "Zero address");
        owner = newOwner;
    }

    // ───────────────────────────── Views ─────────────────────────────────
    /// @notice Full market struct: (question, creator, deadline, yesPool, noPool, outcome, resolved).
    function getMarket(uint256 marketId) external view marketExists(marketId) returns (Market memory) {
        return markets[marketId];
    }

    function bettorCount(uint256 marketId) external view marketExists(marketId) returns (uint256) {
        return bettors[marketId].length;
    }

    /// @notice Implied probability of YES in basis points (5000 = 50%), Polymarket-style "price".
    function yesProbabilityBps(uint256 marketId) external view marketExists(marketId) returns (uint256) {
        Market storage m = markets[marketId];
        uint256 total = m.yesPool + m.noPool;
        if (total == 0) return 5_000;
        return (m.yesPool * BPS) / total;
    }

    /// @notice What `user` would receive if the market resolved to `outcome` right now (after fees).
    function previewPayout(uint256 marketId, address user, bool yesWins)
        external
        view
        marketExists(marketId)
        returns (uint256)
    {
        Market storage m = markets[marketId];
        uint256 total = m.yesPool + m.noPool;
        uint256 winningPool = yesWins ? m.yesPool : m.noPool;
        uint256 stake = yesWins ? yesStake[marketId][user] : noStake[marketId][user];
        if (winningPool == 0 || stake == 0) return 0;
        uint256 distributable = total - (total * (platformFeeBps + creatorFeeBps)) / BPS;
        return (stake * distributable) / winningPool;
    }

    function getBettors(uint256 marketId) external view marketExists(marketId) returns (address[] memory) {
        return bettors[marketId];
    }

    // ──────────────────────────── Internals ──────────────────────────────

    /// @dev Push ETH with a gas cap; if the recipient rejects it, credit a pull balance instead.
    function _pay(uint256 marketId, address to, uint256 amount) private {
        paidOut[marketId][to] += amount;
        (bool ok, ) = payable(to).call{value: amount, gas: 30_000}("");
        if (ok) {
            emit Payout(marketId, to, amount);
        } else {
            pendingWithdrawals[to] += amount;
            emit PayoutDeferred(to, amount);
        }
    }

    function _sendFee(uint256 marketId, address to, uint256 amount, bool isPlatform) private {
        (bool ok, ) = payable(to).call{value: amount, gas: 30_000}("");
        if (ok) {
            emit FeePaid(marketId, to, amount, isPlatform);
        } else {
            pendingWithdrawals[to] += amount;
            emit PayoutDeferred(to, amount);
        }
    }
}
"""

with open("TobesureMarket.sol", "w") as f:
    f.write(CONTRACT_SOURCE)
print("Lines of Solidity:", CONTRACT_SOURCE.count("\n"))

Lines of Solidity: 273


## 3 · Compile

In [5]:
solcx.install_solc(SOLC_VERSION)
compiled = solcx.compile_standard(
    {
        "language": "Solidity",
        "sources": {"TobesureMarket.sol": {"content": CONTRACT_SOURCE}},
        "settings": {
            "optimizer": {"enabled": True, "runs": 200},
            "outputSelection": {"*": {"*": ["abi", "evm.bytecode.object"]}},
        },
    },
    solc_version=SOLC_VERSION,
)
ABI = compiled["contracts"]["TobesureMarket.sol"]["TobesureMarket"]["abi"]
BYTECODE = compiled["contracts"]["TobesureMarket.sol"]["TobesureMarket"]["evm"]["bytecode"]["object"]

with open("TobesureMarket_abi.json", "w") as f:
    json.dump(ABI, f, indent=2)
print("Compiled ✓  functions:", sorted(x["name"] for x in ABI if x["type"] == "function"))
print("Bytecode size:", len(BYTECODE) // 2, "bytes")

Compiled ✓  functions: ['BPS', 'MAX_BETTORS', 'MAX_FEE_BPS', 'MIN_BET', 'bettorCount', 'createMarket', 'creatorFeeBps', 'getBettors', 'getMarket', 'marketCount', 'markets', 'noStake', 'owner', 'paidOut', 'pendingWithdrawals', 'placeBet', 'platformFeeBps', 'previewPayout', 'resolveMarket', 'setFees', 'transferOwnership', 'withdraw', 'yesProbabilityBps', 'yesStake']
Bytecode size: 7914 bytes


## 4 · Transaction helper
Builds, signs and sends a transaction, waits for the receipt, prints the Etherscan link and records everything in `tx_log` (used for the slides / documentation).

In [6]:
tx_log = []

def send_tx(fn_or_tx, account, value_eth=0, label=""):
    """Sign & send a contract call (or deployment) from `account`; returns the receipt."""
    params = {
        "from": account.address,
        "nonce": w3.eth.get_transaction_count(account.address, "pending"),
        "value": w3.to_wei(value_eth, "ether"),
        "chainId": CHAIN_ID,
    }
    tx = fn_or_tx.build_transaction(params)
    tx["gas"] = int(w3.eth.estimate_gas(tx) * 1.2)          # 20% safety buffer
    signed = account.sign_transaction(tx)
    raw = getattr(signed, "raw_transaction", None) or getattr(signed, "rawTransaction")
    tx_hash = w3.eth.send_raw_transaction(raw)
    receipt = w3.eth.wait_for_transaction_receipt(tx_hash, timeout=300)
    status = "✅" if receipt.status == 1 else "❌ REVERTED"
    link = f"{EXPLORER}/tx/{tx_hash.hex() if tx_hash.hex().startswith('0x') else '0x' + tx_hash.hex()}"
    print(f"{status} {label} | gas used {receipt.gasUsed:,} | block {receipt.blockNumber}")
    if NETWORK_MODE == "sepolia":
        print("   ", link)
    tx_log.append({
        "step": label, "from": account.address, "value_eth": value_eth,
        "tx_hash": "0x" + tx_hash.hex().removeprefix("0x"), "block": receipt.blockNumber,
        "gas_used": receipt.gasUsed, "status": receipt.status,
        "etherscan": link if NETWORK_MODE == "sepolia" else "(local)",
    })
    assert receipt.status == 1, f"Transaction failed: {label}"
    return receipt

def eth(wei):
    return float(w3.from_wei(wei, "ether"))

## 5 · Deploy (or reuse an existing deployment)
Constructor arguments: `platformFeeBps = 200` (2%), `creatorFeeBps = 100` (1%).
If `CONTRACT_ADDRESS` is set in `keys.env`, the notebook **reuses** it — that's how we pre-deploy before class and only demo the bet + resolution live.

In [7]:
PLATFORM_FEE_BPS, CREATOR_FEE_BPS = 200, 100
existing = os.getenv("CONTRACT_ADDRESS", "").strip() if NETWORK_MODE == "sepolia" else ""

if existing:
    CONTRACT_ADDRESS = Web3.to_checksum_address(existing)
    print("Reusing deployed contract:", CONTRACT_ADDRESS)
else:
    factory = w3.eth.contract(abi=ABI, bytecode=BYTECODE)
    rcpt = send_tx(factory.constructor(PLATFORM_FEE_BPS, CREATOR_FEE_BPS), deployer, label="Deploy TobesureMarket")
    CONTRACT_ADDRESS = rcpt.contractAddress
    print("\n📄 Contract deployed at:", CONTRACT_ADDRESS)
    if NETWORK_MODE == "sepolia":
        print("   ", f"{EXPLORER}/address/{CONTRACT_ADDRESS}")
        print("   → paste this into keys.env as CONTRACT_ADDRESS=... so later runs reuse it")

market = w3.eth.contract(address=CONTRACT_ADDRESS, abi=ABI)
print("Owner (platform):", market.functions.owner().call())
print("Fees:", market.functions.platformFeeBps().call() / 100, "% platform +",
      market.functions.creatorFeeBps().call() / 100, "% creator")

✅ Deploy TobesureMarket | gas used 12,300,451 | block 12050936
    https://sepolia.etherscan.io/tx/0xfc8ea73fd81dc019824299d6c6b45fde51e5f3bcfb6b7ee01a7f4b6af1a8e7cb

📄 Contract deployed at: 0xe4cE7A7C30241596A9B21DF477A1CcC518c54b6F
    https://sepolia.etherscan.io/address/0xe4cE7A7C30241596A9B21DF477A1CcC518c54b6F
   → paste this into keys.env as CONTRACT_ADDRESS=... so later runs reuse it
Owner (platform): 0xd4556175F88cc8B8612177a771E75AacAB27eff9
Fees: 2.0 % platform + 1.0 % creator


## 6 · Group members' wallets
Each member bets from their **own** wallet. Two ways:
- **A — one laptop:** add `MEMBER_n_NAME` / `MEMBER_n_PRIVATE_KEY` to `keys.env` (fresh testnet-only wallets).
- **B — own laptops:** each member runs this notebook with their own `keys.env` (their key as `MEMBER_1_PRIVATE_KEY`) and the shared `CONTRACT_ADDRESS`, then runs only sections 1, 3, 4, 5, 6 and 8.

In **local** mode, five demo wallets are generated and funded automatically.

In [8]:
members = []
if NETWORK_MODE == "sepolia":
    for i in range(1, 9):
        key = load_key(f"MEMBER_{i}_PRIVATE_KEY")
        if key:
            members.append({"name": os.getenv(f"MEMBER_{i}_NAME", f"Member {i}"), "acct": Account.from_key(key)})
    if not members:   # fall back to the deployer so the flow still runs
        members.append({"name": "Deployer", "acct": deployer})
else:
    for i, n in enumerate(["Member 1", "Member 2", "Member 3", "Member 4", "Member 5"], 1):
        a = Account.create()
        w3.eth.send_transaction({"from": w3.eth.accounts[0], "to": a.address, "value": w3.to_wei(1, "ether")})
        members.append({"name": n, "acct": a})

pd.DataFrame([{"member": m["name"], "address": m["acct"].address,
               "balance_ETH": eth(w3.eth.get_balance(m["acct"].address))} for m in members])

,member,address,balance_ETH
0,Max,0xe29F54b272Aa949f73bEB279bAD7aFA3b2438Bca,0.0
1,Ben,0x65c7985712672d714c74889e18A0a7f5580C3cb9,0.0
2,Ale,0x99FfF1EE1b20770cbcBe8F08EC898a2880B278bE,0.0
3,Sarita,0xbb47A8B156e552FF2793Bf81d17b974c6ed4B434,0.0
4,Anthony,0xf43b78e1380F63C10E5035524D0d675F21dC3Ae5,0.0


### (Optional) Top up members from the deployer
Faucets are rate-limited; the deployer can send each member a little SepoliaETH for gas + bets.

In [13]:
TOP_UP = False          # set True to run
TOP_UP_ETH = 0.005

def transfer(account, to, amount_eth, label):
    tx = {"to": to, "value": w3.to_wei(amount_eth, "ether"), "chainId": CHAIN_ID,
          "nonce": w3.eth.get_transaction_count(account.address, "pending"), "gas": 21_000,
          "maxPriorityFeePerGas": w3.to_wei(1, "gwei"),
          "maxFeePerGas": max(w3.eth.gas_price * 2, w3.to_wei(2, "gwei"))}
    signed = account.sign_transaction(tx)
    raw = getattr(signed, "raw_transaction", None) or getattr(signed, "rawTransaction")
    r = w3.eth.wait_for_transaction_receipt(w3.eth.send_raw_transaction(raw))
    print("✅", label, r.transactionHash.hex())

if TOP_UP:
    for m in members:
        if m["acct"].address != deployer.address:
            transfer(deployer, m["acct"].address, TOP_UP_ETH, f"Top-up {m['name']}")

✅ Top-up Max 66fe1e2de54dad35995d5c6d3d1b93a09981262735d702a4f27b8f519b29e89b
✅ Top-up Ben 1c27960920933e67a444bdd89dac6372b84c598419706f1c61717d0df70152a4
✅ Top-up Ale 7b59126a4aa0178a05933f4e5037c3279e6f473470926a45c3eac2f2165b3a82
✅ Top-up Sarita 341842f30a43be7ab025f92ae78b2b79222ec9bb3d4a2d906a7ea070b9dae856
✅ Top-up Anthony f17b5c6ea3f5eb1afe58d4be715333a929051ea454559080e3a65af851888123


## 7 · Create a prediction market

In [11]:
QUESTION = "Will ETH close above $4,000 on 30 November 2026?"
BETTING_WINDOW_SECONDS = 7 * 24 * 3600     # bets open for 7 days

rcpt = send_tx(market.functions.createMarket(QUESTION, BETTING_WINDOW_SECONDS), deployer, label="Create market")
event = market.events.MarketCreated().process_receipt(rcpt, errors=DISCARD)[0]["args"]
MARKET_ID = event["marketId"]
print(f"\n🆕 Market #{MARKET_ID}: {event['question']}")
print("   Creator:", event["creator"])
print("   Betting closes:", datetime.fromtimestamp(event["deadline"], tz=timezone.utc).strftime("%Y-%m-%d %H:%M UTC"))

✅ Create market | gas used 681,267 | block 12050989
    https://sepolia.etherscan.io/tx/0xc4020851a51fb7cdadeb084cd072ed8b1aa4e41273e15955ee9b8a269dcee1dd

🆕 Market #0: Will ETH close above $4,000 on 30 November 2026?
   Creator: 0xd4556175F88cc8B8612177a771E75AacAB27eff9
   Betting closes: 2026-11-09 22:00 UTC


## 8 · Every group member places a bet
Edit `BETS` — `True` = YES, `False` = NO. (If you joined via option B, set `MARKET_ID` to the shared market id first.)

In [14]:
# MARKET_ID = 0     # option B: uncomment and set the shared market id
BETS = [
    (True,  0.002),   # Max      Yes
    (False, 0.003),   # Ben      No
    (True,  0.001),   # Ale      Yes
    (False, 0.001),   # Sarita   No
    (True,  0.002),   # Anthony  Yes
]

for m, (side, amount) in zip(members, BETS):
    send_tx(market.functions.placeBet(MARKET_ID, side), m["acct"], value_eth=amount,
            label=f"Bet {'YES' if side else 'NO'} {amount} ETH — {m['name']}")

Web3RPCError: {'code': -32000, 'message': 'failed with 200000000 gas: insufficient funds for gas * price + value: address 0xe29F54b272Aa949f73bEB279bAD7aFA3b2438Bca have 0 want 2000000000000000'}

In [16]:
import time

def top_up(to, amount_eth, label):
    tx = {"to": to, "value": w3.to_wei(amount_eth, "ether"), "chainId": CHAIN_ID,
          "nonce": w3.eth.get_transaction_count(deployer.address, "pending"),
          "maxPriorityFeePerGas": w3.to_wei(1, "gwei"),
          "maxFeePerGas": max(w3.eth.gas_price * 2, w3.to_wei(2, "gwei"))}
    tx["gas"] = int(w3.eth.estimate_gas({**tx, "from": deployer.address}) * 1.5)
    signed = deployer.sign_transaction(tx)
    raw = getattr(signed, "raw_transaction", None) or getattr(signed, "rawTransaction")
    r = w3.eth.wait_for_transaction_receipt(w3.eth.send_raw_transaction(raw), timeout=300)
    ok = "✅" if r.status == 1 else "❌ FEHLGESCHLAGEN"
    print("   ", ok, label, "| Block", r.blockNumber, "| 0x" + r.transactionHash.hex().removeprefix("0x"))

def wait_for_balance(addr, minimum_eth, timeout=120):
    start = time.time()
    while True:
        bal = eth(w3.eth.get_balance(addr))
        if bal >= minimum_eth or time.time() - start > timeout:
            return bal
        time.sleep(5)

print("Aktueller Block:", w3.eth.block_number)
print("Platform:", round(eth(w3.eth.get_balance(deployer.address)), 6), "ETH\n")
print("Kontostände:")
for m in members:
    addr = m["acct"].address
    bal = eth(w3.eth.get_balance(addr))
    code = len(w3.eth.get_code(addr))
    print(f"  {m['name']}: {bal:.6f} ETH  ({addr[:8]}…{addr[-4:]}, Code-Bytes: {code})")
    if bal < 0.004:
        top_up(addr, 0.005, f"Top-up {m['name']}")

print("\nWetten:")
for m, (side, amount) in zip(members, BETS):
    addr = m["acct"].address
    already = market.functions.yesStake(MARKET_ID, addr).call() + market.functions.noStake(MARKET_ID, addr).call()
    if already > 0:
        print(f"  {m['name']} hat schon gewettet, übersprungen")
        continue
    bal = wait_for_balance(addr, amount + 0.0005)
    if bal < amount + 0.0005:
        print(f"  ⚠️ {m['name']} hat nach 2 Minuten nur {bal:.6f} ETH, Wette übersprungen")
        continue
    send_tx(market.functions.placeBet(MARKET_ID, side), m["acct"], value_eth=amount,
            label=f"Bet {'YES' if side else 'NO'} {amount} ETH — {m['name']}")

Aktueller Block: 12051075
Platform: 0.049685 ETH

Kontostände:
  Max: 0.000000 ETH  (0xe29F54…8Bca, Code-Bytes: 0)
    ✅ Top-up Max | Block 12051076 | 0xa3ba2587b19b4142f4786bb81d8320f3ef56d9e81174dc952d1c86de172e92d7
  Ben: 0.000000 ETH  (0x65c798…3cb9, Code-Bytes: 0)
    ✅ Top-up Ben | Block 12051077 | 0x1f803905f3e3521e834af0c888d0a78ea245949436f28af070bacc90551475fe
  Ale: 0.000000 ETH  (0x99FfF1…78bE, Code-Bytes: 0)
    ✅ Top-up Ale | Block 12051078 | 0x750b1e429a7b4ecdad96ab3dc8b6730ee9af74f4040ee6001357d036efc5b4ce
  Sarita: 0.000000 ETH  (0xbb47A8…B434, Code-Bytes: 0)
    ✅ Top-up Sarita | Block 12051079 | 0xd798799b09a06c486fc9682de26b276f0f1d17291b69a8ec03104a8f3110b532
  Anthony: 0.000000 ETH  (0xf43b78…3Ae5, Code-Bytes: 0)
    ✅ Top-up Anthony | Block 12051080 | 0xd9d4a083910625f0c7829bf7044953e8b212f308cb74ac13c511cb2671e1e4f1

Wetten:
✅ Bet YES 0.002 ETH — Max | gas used 582,037 | block 12051081
    https://sepolia.etherscan.io/tx/0x03d27a16f8c08a813bbc5ec217d045d300000be

## 9 · Market state — pools, implied odds, payout preview

In [17]:
def market_snapshot(mid):
    q, creator, deadline, yes_pool, no_pool, outcome, resolved = market.functions.getMarket(mid).call()
    prob = market.functions.yesProbabilityBps(mid).call() / 100
    print(f"Market #{mid}: {q}")
    print(f"  YES pool {eth(yes_pool):.4f} ETH | NO pool {eth(no_pool):.4f} ETH | total {eth(yes_pool + no_pool):.4f} ETH")
    print(f"  Implied P(YES) = {prob:.1f}%   |   resolved: {resolved}   outcome: {['Unresolved','YES','NO','INVALID'][outcome]}")
    rows = []
    for addr in market.functions.getBettors(mid).call():
        name = next((m["name"] for m in members if m["acct"].address == addr), addr[:10] + "…")
        rows.append({
            "member": name,
            "YES stake": eth(market.functions.yesStake(mid, addr).call()),
            "NO stake": eth(market.functions.noStake(mid, addr).call()),
            "payout if YES": eth(market.functions.previewPayout(mid, addr, True).call()),
            "payout if NO": eth(market.functions.previewPayout(mid, addr, False).call()),
        })
    return pd.DataFrame(rows)

market_snapshot(MARKET_ID)

Market #0: Will ETH close above $4,000 on 30 November 2026?
  YES pool 0.0050 ETH | NO pool 0.0040 ETH | total 0.0090 ETH
  Implied P(YES) = 55.5%   |   resolved: False   outcome: Unresolved


,member,YES stake,NO stake,payout if YES,payout if NO
0,Max,0.002,0.000,0.003492,0.000000
1,Ben,0.000,0.003,0.000000,0.006548
2,Ale,0.001,0.000,0.001746,0.000000
3,Sarita,0.000,0.001,0.000000,0.002182
4,Anthony,0.002,0.000,0.003492,0.000000


## 10 · 🎤 LIVE DEMO — one more bet in class
Run this live during the presentation (the earlier bets can be pre-done).

In [18]:
LIVE_BETTOR = members[0]          # whoever is presenting
LIVE_SIDE, LIVE_AMOUNT = True, 0.001

send_tx(market.functions.placeBet(MARKET_ID, LIVE_SIDE), LIVE_BETTOR["acct"], value_eth=LIVE_AMOUNT,
        label=f"LIVE bet {'YES' if LIVE_SIDE else 'NO'} — {LIVE_BETTOR['name']}")
market_snapshot(MARKET_ID)

✅ LIVE bet YES — Max | gas used 57,297 | block 12051141
    https://sepolia.etherscan.io/tx/0x658b0d8a7a935bb69cd0eb882d8e65dc9a54628c56612858c41b26f6928f4405
Market #0: Will ETH close above $4,000 on 30 November 2026?
  YES pool 0.0060 ETH | NO pool 0.0040 ETH | total 0.0100 ETH
  Implied P(YES) = 60.0%   |   resolved: False   outcome: Unresolved


,member,YES stake,NO stake,payout if YES,payout if NO
0,Max,0.003,0.000,0.004850,0.000000
1,Ben,0.000,0.003,0.000000,0.007275
2,Ale,0.001,0.000,0.001617,0.000000
3,Sarita,0.000,0.001,0.000000,0.002425
4,Anthony,0.002,0.000,0.003233,0.000000


## 11 · 🎤 LIVE DEMO — resolve the market & distribute payouts
Outcome codes: `1 = YES`, `2 = NO`, `3 = INVALID (refund everyone)`. Only the market creator (or platform owner) can call this.

In [19]:
OUTCOME = 1    # 1 = YES wins

watch = {m["name"]: m["acct"].address for m in members}
owner_addr = market.functions.owner().call()
if owner_addr == deployer.address:
    watch["Deployer (platform owner + market creator)"] = deployer.address
else:
    watch["Platform (owner)"] = owner_addr
    watch["Market creator"] = deployer.address
before = {k: w3.eth.get_balance(v) for k, v in watch.items()}

rcpt = send_tx(market.functions.resolveMarket(MARKET_ID, OUTCOME), deployer, label=f"Resolve market #{MARKET_ID}")

res = market.events.MarketResolved().process_receipt(rcpt, errors=DISCARD)[0]["args"]
print(f"\n🏁 Outcome: {['Unresolved','YES','NO','INVALID'][res['outcome']]} | pool {eth(res['totalPool']):.4f} ETH "
      f"| platform fee {eth(res['platformFee']):.6f} | creator fee {eth(res['creatorFee']):.6f}")

payouts = market.events.Payout().process_receipt(rcpt, errors=DISCARD)
fees = market.events.FeePaid().process_receipt(rcpt, errors=DISCARD)
name_of = {v: k for k, v in watch.items()}
rows = [{"recipient": name_of.get(e["args"]["to"], e["args"]["to"]), "type": "winnings / refund",
         "amount_ETH": eth(e["args"]["amount"])} for e in payouts]
rows += [{"recipient": name_of.get(e["args"]["to"], e["args"]["to"]),
          "type": "platform fee" if e["args"]["isPlatformFee"] else "creator fee",
          "amount_ETH": eth(e["args"]["amount"])} for e in fees]
pd.DataFrame(rows)

✅ Resolve market #0 | gas used 570,481 | block 12051154
    https://sepolia.etherscan.io/tx/0xd871b2b3afc05d389504293d4790cca848c9f22146f7ae3e8e1fcb133b577162

🏁 Outcome: YES | pool 0.0100 ETH | platform fee 0.000200 | creator fee 0.000100


,recipient,type,amount_ETH
0,Max,winnings / refund,0.004850
1,Ale,winnings / refund,0.001617
2,Anthony,winnings / refund,0.003233
3,Deployer (platform owner + market creator),creator fee,0.000100
4,Deployer (platform owner + market creator),platform fee,0.000200


## 12 · Verification

In [20]:
gas_cost = rcpt.gasUsed * rcpt.effectiveGasPrice
summary = []
for k, addr in watch.items():
    delta = w3.eth.get_balance(addr) - before[k]
    if addr == deployer.address:
        delta += gas_cost            # deployer paid the resolve gas
    summary.append({"wallet": k, "balance change (ETH, excl. resolve gas)": round(eth(delta), 8)})
display(pd.DataFrame(summary))

contract_balance = w3.eth.get_balance(CONTRACT_ADDRESS)
pending = sum(market.functions.pendingWithdrawals(a).call() for a in set(watch.values()))
print("Contract balance after resolution:", eth(contract_balance), "ETH (should equal pending withdrawals:", eth(pending), ")")
print("All funds distributed ✓" if contract_balance == pending else "⚠️ check balances")
market_snapshot(MARKET_ID)

,wallet,"balance change (ETH, excl. resolve gas)"
0,Max,0.004850
1,Ben,0.000000
2,Ale,0.001617
3,Sarita,0.000000
4,Anthony,0.003233
5,Deployer (platform owner + market creator),0.000300


Contract balance after resolution: 0.0 ETH (should equal pending withdrawals: 0.0 )
All funds distributed ✓
Market #0: Will ETH close above $4,000 on 30 November 2026?
  YES pool 0.0060 ETH | NO pool 0.0040 ETH | total 0.0100 ETH
  Implied P(YES) = 60.0%   |   resolved: True   outcome: YES


,member,YES stake,NO stake,payout if YES,payout if NO
0,Max,0.003,0.000,0.004850,0.000000
1,Ben,0.000,0.003,0.000000,0.007275
2,Ale,0.001,0.000,0.001617,0.000000
3,Sarita,0.000,0.001,0.000000,0.002425
4,Anthony,0.002,0.000,0.003233,0.000000


## 13 · Security checks (expected to fail ✋)
These calls **must revert** — they prove the access-control and lifecycle rules work. Uses `.call()` so no gas is spent.

In [21]:
def expect_revert(desc, fn, sender, value_eth=0):
    try:
        fn.call({"from": sender, "value": w3.to_wei(value_eth, "ether")})
        print(f"❌ {desc}: did NOT revert")
    except Exception as e:
        print(f"✅ {desc}: reverted → {str(e)[:70]}")

outsider = members[-1]["acct"].address if members[-1]["acct"].address != deployer.address else Account.create().address
expect_revert("Bet on a resolved market", market.functions.placeBet(MARKET_ID, True), outsider, 0.001)
expect_revert("Resolve twice", market.functions.resolveMarket(MARKET_ID, 2), deployer.address)
expect_revert("Non-creator resolves", market.functions.resolveMarket(MARKET_ID, 1), outsider)
expect_revert("Bet on a non-existent market", market.functions.placeBet(999, True), outsider, 0.001)
expect_revert("Fees above 10% cap", market.functions.setFees(800, 300), deployer.address)

✅ Bet on a resolved market: reverted → ('execution reverted: Market resolved', '0x08c379a00000000000000000000
✅ Resolve twice: reverted → ('execution reverted: Already resolved', '0x08c379a0000000000000000000
✅ Non-creator resolves: reverted → ('execution reverted: Not creator or owner', '0x08c379a000000000000000
✅ Bet on a non-existent market: reverted → ('execution reverted: Market does not exist', '0x08c379a00000000000000
✅ Fees above 10% cap: reverted → ('execution reverted: Fees too high', '0x08c379a0000000000000000000000


## 14 · Transaction log for the presentation

In [22]:
log = pd.DataFrame(tx_log)
log.to_csv("tx_log.csv", index=False)
print("Contract:", f"{EXPLORER}/address/{CONTRACT_ADDRESS}" if NETWORK_MODE == "sepolia" else CONTRACT_ADDRESS)
print("Saved tx_log.csv — paste the Etherscan links into the 'Deployment & transactions' slide.\n")
for r in tx_log:
    print(f"- {r['step']}: {r['etherscan']}")
log[["step", "tx_hash", "block", "gas_used", "status"]]

Contract: https://sepolia.etherscan.io/address/0xe4cE7A7C30241596A9B21DF477A1CcC518c54b6F
Saved tx_log.csv — paste the Etherscan links into the 'Deployment & transactions' slide.

- Deploy TobesureMarket: https://sepolia.etherscan.io/tx/0xfc8ea73fd81dc019824299d6c6b45fde51e5f3bcfb6b7ee01a7f4b6af1a8e7cb
- Create market: https://sepolia.etherscan.io/tx/0xc4020851a51fb7cdadeb084cd072ed8b1aa4e41273e15955ee9b8a269dcee1dd
- Bet YES 0.002 ETH — Max: https://sepolia.etherscan.io/tx/0x03d27a16f8c08a813bbc5ec217d045d300000be62e7525e0884204c33b05f97d
- Bet NO 0.003 ETH — Ben: https://sepolia.etherscan.io/tx/0xa23ddfe820fff1f2c1e3a682c49b8b6316ab8887e8bc663f239de81853f83379
- Bet YES 0.001 ETH — Ale: https://sepolia.etherscan.io/tx/0x4c1eff51ee799366d6d3738e029dced9f9901cccd675b73e74e1dbeac69ff873
- Bet NO 0.001 ETH — Sarita: https://sepolia.etherscan.io/tx/0x08ecc26f17eaa82d812b1afc8888ded974d185fe84b4f1f47f8c71ab8f72f6ac
- Bet YES 0.002 ETH — Anthony: https://sepolia.etherscan.io/tx/0xfefecb0cde

,step,tx_hash,block,gas_used,status
0,Deploy TobesureMarket,0xfc8ea73fd81dc019824299d6c6b45fde51e5f3bcfb6b...,12050936,12300451,1
1,Create market,0xc4020851a51fb7cdadeb084cd072ed8b1aa4e41273e1...,12050989,681267,1
2,Bet YES 0.002 ETH — Max,0x03d27a16f8c08a813bbc5ec217d045d300000be62e75...,12051081,582037,1
3,Bet NO 0.003 ETH — Ben,0xa23ddfe820fff1f2c1e3a682c49b8b6316ab8887e8bc...,12051082,484092,1
4,Bet YES 0.001 ETH — Ale,0x4c1eff51ee799366d6d3738e029dced9f9901cccd675...,12051083,386197,1
5,Bet NO 0.001 ETH — Sarita,0x08ecc26f17eaa82d812b1afc8888ded974d185fe84b4...,12051084,386172,1
6,Bet YES 0.002 ETH — Anthony,0xfefecb0cde713d321ae5e5f51c10c36d3c2444bfd402...,12051085,386197,1
7,LIVE bet YES — Max,0x658b0d8a7a935bb69cd0eb882d8e65dc9a54628c5661...,12051141,57297,1
8,Resolve market #0,0xd871b2b3afc05d389504293d4790cca848c9f22146f7...,12051154,570481,1


## 15 · Reflection — challenges & mainnet readiness
1. **Oracle / trusted resolver** – the creator decides the outcome. Mainnet: UMA Optimistic Oracle (what Polymarket uses) or Chainlink, plus a dispute window.
2. **Pari-mutuel vs order book** – our odds are only known at resolution. Polymarket uses ERC-1155 outcome shares (Gnosis CTF) traded on a CLOB, so prices = live probabilities and users can exit early.
3. **Gas & scalability** – the payout loop is capped at 200 bettors. Mainnet: pull-based `claim()` per user, or an L2 (Polymarket runs on Polygon).
4. **Stable collateral** – ETH price risk distorts bets; use USDC.
5. **Security** – external audit, OpenZeppelin `ReentrancyGuard`/`Ownable2Step`, multisig owner, pausability, formal tests (Foundry fuzzing).
6. **Regulation & UX** – prediction markets are regulated (CFTC in the US, gambling law in the EU), so KYC/geo-fencing; plus a frontend with wallet connect.